# Automatización y Limpieza de Reportes Excel/CSV con Python y Pandas
**Plantilla práctica de ProgramBI** — Limpieza de montos en pesos chilenos (`$1.250.000`, `CLP 450.000`), normalización de fechas mixtas (`DD/MM/YYYY` y `YYYY-MM-DD`), deduplicación, cálculo de KPIs y exportación a Excel profesional con múltiples hojas.

In [ ]:
import re
import pandas as pd
from pathlib import Path

# Cargar el archivo sucio de ejemplo
ruta_csv = Path('ventas-sucias-ejemplo.csv')
df_raw = pd.read_csv(ruta_csv, dtype=str)
print(f'Filas originales: {len(df_raw)}')
df_raw.head(10)

In [ ]:
# 1. Limpieza de strings de texto (espacios extras y capitalización)
df = df_raw.copy()
for col in ['Sucursal', 'Vendedor', 'Categoria', 'Estado']:
    df[col] = df[col].fillna('Sin definir').str.strip().str.title()

# 2. Función robusta para limpiar montos CLP sucios ('$ 1.250.000', 'CLP 450.000', '320000')
def limpiar_clp(valor: str) -> float:
    if pd.isna(valor):
        return 0.0
    limpio = re.sub(r'[^0-9,-]', '', str(valor))
    limpio = limpio.replace(',', '.')
    try:
        return float(limpio)
    except ValueError:
        return 0.0

df['Venta_CLP'] = df['Monto_Bruto'].apply(limpiar_clp)
df['Costo_CLP'] = df['Costo'].apply(limpiar_clp)
df['Unidades'] = pd.to_numeric(df['Unidades'], errors='coerce').fillna(0).astype(int)

In [ ]:
# 3. Normalización de fechas mixtas (DD/MM/YYYY y YYYY-MM-DD)
df['Fecha_Limpia'] = pd.to_datetime(df['Fecha'], format='mixed', dayfirst=True, errors='coerce')
df = df.dropna(subset=['Fecha_Limpia']).copy()
df['Anio_Mes'] = df['Fecha_Limpia'].dt.to_period('M').astype(str)

# 4. Eliminar filas duplicadas exactas y filtrar solo ventas válidas
antes = len(df)
df = df.drop_duplicates(subset=['ID_Venta', 'Fecha_Limpia', 'Sucursal', 'Venta_CLP'])
df = df[df['Estado'] != 'Anulado'].copy()
print(f'Duplicados/anulados removidos: {antes - len(df)} | Filas limpias: {len(df)}')

In [ ]:
# 5. Cálculo de métricas de negocio y tablas resumen
df['Margen_CLP'] = df['Venta_CLP'] - df['Costo_CLP']
df['Margen_Pct'] = (df['Margen_CLP'] / df['Venta_CLP'].replace(0, pd.NA)).fillna(0).round(4)

resumen_sucursal = (
    df.groupby('Sucursal', as_index=False)
      .agg(Ventas_CLP=('Venta_CLP', 'sum'),
           Margen_CLP=('Margen_CLP', 'sum'),
           Pedidos=('ID_Venta', 'count'))
      .sort_values('Ventas_CLP', ascending=False)
)
resumen_sucursal['Margen_Pct'] = (resumen_sucursal['Margen_CLP'] / resumen_sucursal['Ventas_CLP']).round(4)

resumen_mensual = (
    df.groupby('Anio_Mes', as_index=False)
      .agg(Ventas_CLP=('Venta_CLP', 'sum'),
           Margen_CLP=('Margen_CLP', 'sum'),
           Unidades=('Unidades', 'sum'))
      .sort_values('Anio_Mes')
)
resumen_sucursal

In [ ]:
# 6. Exportar reporte consolidado a Excel con múltiples pestañas
salida = Path('reporte_ventas_consolidado.xlsx')
with pd.ExcelWriter(salida, engine='openpyxl') as writer:
    resumen_sucursal.to_excel(writer, sheet_name='Resumen_Sucursal', index=False)
    resumen_mensual.to_excel(writer, sheet_name='Evolucion_Mensual', index=False)
    df.to_excel(writer, sheet_name='Base_Limpia', index=False)

print(f'Reporte exportado correctamente en: {salida.resolve()}')